# 🏢 Zyro Dynamics HR Help Desk — RAG Challenge
**Complete RAG pipeline: Load PDFs → Chunk → FAISS → Groq LLM → Guardrails → Submission**

## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q langchain langchain-community langchain-groq langchain-huggingface \
    faiss-cpu pypdf sentence-transformers langsmith python-dotenv streamlit

## Cell 2 — Imports & Configuration

In [ ]:
import os
import re
import json
import base64
import csv
from pathlib import Path

# LangChain
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.schema import Document

# Kaggle secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GROQ_API_KEY     = secrets.get_secret('GROQ_API_KEY')
LANGCHAIN_API_KEY = secrets.get_secret('LANGCHAIN_API_KEY')

os.environ['GROQ_API_KEY']          = GROQ_API_KEY
os.environ['LANGCHAIN_API_KEY']     = LANGCHAIN_API_KEY
os.environ['LANGCHAIN_TRACING_V2']  = 'true'
os.environ['LANGCHAIN_PROJECT']     = 'zyro-rag-challenge'
os.environ['LANGCHAIN_ENDPOINT']    = 'https://api.smith.langchain.com'

# Config
PDF_DIR          = Path('/kaggle/input/niat-masterclass-rag-challenge')
CHUNK_SIZE       = 800
CHUNK_OVERLAP    = 150
TOP_K            = 5
EMBEDDING_MODEL  = 'sentence-transformers/all-MiniLM-L6-v2'
LLM_MODEL        = 'llama-3.3-70b-versatile'

print('✅ Configuration complete')
print(f'   PDF directory : {PDF_DIR}')
print(f'   LLM model     : {LLM_MODEL}')
print(f'   Embedding     : {EMBEDDING_MODEL}')

## Cell 3 — TODO 1: Load HR PDFs

In [ ]:
# TODO 1 — Load all HR policy PDFs from the dataset directory

pdf_files = sorted(PDF_DIR.glob('*.pdf'))
print(f'Found {len(pdf_files)} PDF files:')
for f in pdf_files:
    print(f'  - {f.name}')

all_documents: list[Document] = []

for pdf_path in pdf_files:
    try:
        loader = PyPDFLoader(str(pdf_path))
        pages  = loader.load()
        # Attach source filename to each page's metadata
        for page in pages:
            page.metadata['source_file'] = pdf_path.name
        all_documents.extend(pages)
        print(f'  ✅ Loaded {pdf_path.name} ({len(pages)} pages)')
    except Exception as e:
        print(f'  ❌ Failed to load {pdf_path.name}: {e}')

print(f'\nTotal pages loaded: {len(all_documents)}')

## Cell 4 — TODO 2: Chunk the Documents

In [ ]:
# TODO 2 — Split documents into overlapping chunks for retrieval

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '.', ' ', ''],
    length_function=len,
)

chunks = splitter.split_documents(all_documents)

print(f'Total chunks created : {len(chunks)}')
print(f'Avg chunk length     : {sum(len(c.page_content) for c in chunks)//len(chunks)} chars')
print(f'\nSample chunk (first):')
print('-' * 60)
print(chunks[0].page_content[:400])
print(f'\nMetadata: {chunks[0].metadata}')

## Cell 5 — TODO 3: Build FAISS Vector Store

In [ ]:
# TODO 3 — Embed chunks and build FAISS vector store

print('Loading embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print('Building FAISS index...')
vectorstore = FAISS.from_documents(chunks, embeddings)

# MMR retriever — diverse + relevant results
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': TOP_K, 'fetch_k': 20, 'lambda_mult': 0.6},
)

# Quick sanity check
test_results = retriever.invoke('What is the leave policy?')
print(f'\n✅ FAISS vector store ready!')
print(f'   Indexed {len(chunks)} chunks')
print(f'   Test retrieval returned {len(test_results)} chunks')
for r in test_results[:2]:
    print(f'   Source: {r.metadata.get("source_file","?")} | Page {r.metadata.get("page",0)+1}')

## Cell 6 — TODO 4: Build LCEL RAG Chain

In [ ]:
# TODO 4 — Build the LCEL RAG chain with Groq LLM

# ── LLM ──────────────────────────────────────────────────────────────────────
llm = ChatGroq(
    model=LLM_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0.1,
    max_tokens=1024,
)

# ── Prompt template ───────────────────────────────────────────────────────────
system_prompt = """You are an expert HR Help Desk assistant for Zyro Dynamics Pvt. Ltd.
Your job is to answer employee questions ONLY based on the provided HR policy documents.

Guidelines:
- Answer clearly and accurately using ONLY the retrieved context below.
- If the context does not contain enough information, say so honestly.
- Do NOT fabricate information or use outside knowledge.
- Format lists and steps clearly when appropriate.
- Keep answers concise but complete.

Retrieved Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', '{question}'),
])

# ── Helper: format retrieved docs ─────────────────────────────────────────────
def format_docs(docs: list[Document]) -> str:
    return '\n\n---\n\n'.join(
        f"[Source: {d.metadata.get('source_file','unknown')} | Page {d.metadata.get('page',0)+1}]\n{d.page_content}"
        for d in docs
    )

# ── LCEL RAG chain ────────────────────────────────────────────────────────────
rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x['question']))
    )
    | prompt
    | llm
    | StrOutputParser()
)

print('✅ RAG chain built successfully')

## Cell 7 — TODO 5: Guardrails for Out-of-Scope Detection

In [ ]:
# TODO 5 — Add guardrails for out-of-scope question detection

# Fast keyword-based pre-filter
OUT_OF_SCOPE_KEYWORDS = [
    # Finance / business
    'stock price', 'share price', 'revenue', 'profit', 'quarterly', 'fiscal',
    'market cap', 'investment', 'trading', 'ipo', 'dividend',
    # Competitor / external
    'competitor', 'amazon', 'google', 'microsoft', 'apple',
    # Personal / unrelated
    'weather', 'cricket', 'sports', 'movie', 'recipe', 'cook',
    # Medical / legal / political
    'covid vaccine', 'medicine', 'election', 'politics', 'religion',
    # Personal finance
    'personal loan', 'bank account', 'credit card', 'emi',
]

REFUSAL_RESPONSE = (
    "I'm sorry, but I can only answer HR-related questions based on "
    "Zyro Dynamics' internal policy documents. Your question appears to be "
    "outside the scope of our HR knowledge base. Please reach out to the "
    "appropriate team or department for assistance."
)

# LLM refusal phrases
LLM_REFUSAL_PHRASES = [
    'cannot answer', 'outside my scope', 'not covered',
    'cannot find', 'no information', 'not available in the provided',
    'unable to find', 'not in the hr policies',
]

def is_out_of_scope_keyword(question: str) -> bool:
    """Quick keyword pre-check."""
    q = question.lower()
    return any(kw in q for kw in OUT_OF_SCOPE_KEYWORDS)

def answer_with_guardrails(question: str) -> dict:
    """
    Full RAG pipeline with guardrails.
    Returns: {'question': str, 'answer': str, 'sources': list, 'out_of_scope': bool}
    """
    # 1. Keyword pre-check
    if is_out_of_scope_keyword(question):
        return {
            'question'   : question,
            'answer'     : REFUSAL_RESPONSE,
            'sources'    : [],
            'out_of_scope': True,
        }

    # 2. Retrieve relevant chunks
    source_docs = retriever.invoke(question)

    # 3. Generate answer using RAG chain
    answer = rag_chain.invoke({'question': question})

    # 4. LLM-level out-of-scope detection
    if any(phrase in answer.lower() for phrase in LLM_REFUSAL_PHRASES):
        return {
            'question'   : question,
            'answer'     : REFUSAL_RESPONSE,
            'sources'    : [],
            'out_of_scope': True,
        }

    # Build unique source references
    seen = set()
    sources = []
    for doc in source_docs:
        key = (doc.metadata.get('source_file',''), doc.metadata.get('page',''))
        if key not in seen:
            seen.add(key)
            sources.append({
                'file'   : doc.metadata.get('source_file', 'Unknown'),
                'page'   : doc.metadata.get('page', 0) + 1,
                'snippet': doc.page_content[:200],
            })

    return {
        'question'   : question,
        'answer'     : answer,
        'sources'    : sources,
        'out_of_scope': False,
    }

print('✅ Guardrails configured')
print(f'   Keyword filters : {len(OUT_OF_SCOPE_KEYWORDS)} terms')
print(f'   LLM refusal check: {len(LLM_REFUSAL_PHRASES)} phrases')

## Cell 8 — TODO 6: Test the RAG Bot (Sample Questions)

In [ ]:
# TODO 6 — Test with sample questions (these will appear in LangSmith traces)

SAMPLE_QUESTIONS = [
    'How many casual leaves are employees entitled to per year?',
    'What is the work from home policy at Zyro Dynamics?',
    'How is the annual performance review conducted?',
]

for i, q in enumerate(SAMPLE_QUESTIONS, 1):
    print(f'\n{'='*70}')
    print(f'Q{i}: {q}')
    print('-'*70)
    result = answer_with_guardrails(q)
    print(f'Answer: {result["answer"]}')
    if result['sources']:
        print(f'\nSources ({len(result["sources"])}):')
        for s in result['sources']:
            print(f'  - {s["file"]} | Page {s["page"]}')
    print(f'Out-of-scope: {result["out_of_scope"]}')

## Cell 9 — Answer All 15 Competition Questions

These are the questions Q01-Q15 for the competition submission.

In [ ]:
# All 15 competition questions
# Q01-Q10: In-scope HR questions
# Q11-Q15: Out-of-scope (must refuse!)

COMPETITION_QUESTIONS = [
    # Q01-Q10: HR policy questions
    "How many earned leaves (EL) are employees entitled to per year at Zyro Dynamics?",
    "What is the maternity leave entitlement at Zyro Dynamics?",
    "What is the work from home policy for employees at Zyro Dynamics?",
    "How is the annual performance review conducted and what are the rating categories?",
    "What is the probation period for new employees at Zyro Dynamics?",
    "What are the key components of the compensation and CTC structure at Zyro Dynamics?",
    "What are the IT security guidelines employees must follow regarding devices and data?",
    "What is the process for reporting a POSH (Prevention of Sexual Harassment) complaint?",
    "What are the travel reimbursement and expense claim procedures at Zyro Dynamics?",
    "What are the code of conduct guidelines regarding discipline and ethics at Zyro Dynamics?",
    # Q11-Q15: Out-of-scope questions (must refuse gracefully)
    "What is the current stock price of Zyro Dynamics?",
    "What is the quarterly revenue of Zyro Dynamics for Q3?",
    "Who is the CEO of Apple Inc. and what is their net worth?",
    "What is the recipe for chocolate cake?",
    "What are the latest cricket match scores?",
]

# Collect answers
competition_answers = []
for i, question in enumerate(COMPETITION_QUESTIONS, 1):
    q_id = f'Q{i:02d}'
    print(f'\nAnswering {q_id}: {question[:70]}...')
    result = answer_with_guardrails(question)
    competition_answers.append({
        'question_id': q_id,
        'question'   : question,
        'answer'     : result['answer'],
        'out_of_scope': result['out_of_scope'],
    })
    status = '🚫 REFUSED (out-of-scope)' if result['out_of_scope'] else '✅ Answered'
    print(f'  {status}')

print(f'\n✅ All {len(competition_answers)} questions answered!')

## Cell 10 — Review Answers

In [ ]:
# Review all answers before submission
for item in competition_answers:
    print(f"\n{'='*70}")
    print(f"{item['question_id']}: {item['question']}")
    print(f"{'-'*70}")
    print(f"Answer: {item['answer']}")
    oos = '🚫 OUT-OF-SCOPE' if item['out_of_scope'] else '✅ IN-SCOPE'
    print(f"Status: {oos}")

## Cell 11 — LangSmith Trace Instructions

1. Go to **https://smith.langchain.com**
2. Open project → **zyro-rag-challenge**
3. Click any trace from the test runs above
4. Click **Share** → copy the public URL
5. Paste it in Cell 12 below

## Cell 12 — Streamlit App Code (Cell 14 task)

The complete `app.py` is below. Deploy it on https://share.streamlit.io

In [ ]:
streamlit_app_code = '''
# Paste this as app.py in your Streamlit deployment
# (Full code is in the competition files — app.py)
'''
print('See the app.py file for the complete Streamlit chatbot code.')
print('Deploy steps:')
print('  1. Push app.py + requirements.txt to a public GitHub repo')
print('  2. Go to https://share.streamlit.io → New App')
print('  3. Connect your GitHub repo')
print('  4. Add GROQ_API_KEY in Streamlit Secrets')
print('  5. Copy the deployed URL')

## Cell 13 — Generate submission.csv

In [ ]:
import base64
import csv

# ── FILL THESE IN ─────────────────────────────────────────────────────────────
STREAMLIT_URL  = 'https://your-app.streamlit.app'  # Replace with your Streamlit URL
LANGSMITH_URL  = 'https://smith.langchain.com/public/your-trace-id'  # Replace with your LangSmith trace URL
# ─────────────────────────────────────────────────────────────────────────────

def encode_text(text: str) -> str:
    """Base64-encode a string."""
    return base64.b64encode(text.encode('utf-8')).decode('utf-8')

rows = []
for item in competition_answers:
    rows.append({
        'question_id'   : item['question_id'],
        'question_enc'  : encode_text(item['question']),
        'answer_enc'    : encode_text(item['answer']),
        'streamlit_link': STREAMLIT_URL,
        'langsmith_link': LANGSMITH_URL,
    })

output_file = '/kaggle/working/submission.csv'
with open(output_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['question_id','question_enc','answer_enc','streamlit_link','langsmith_link'])
    writer.writeheader()
    writer.writerows(rows)

print(f'✅ submission.csv generated → {output_file}')
print(f'   Rows: {len(rows)}')
print(f'\nFirst 3 rows preview:')
for r in rows[:3]:
    print(f"  {r['question_id']} | answer_enc: {r['answer_enc'][:40]}...")

# Verify
assert len(rows) == 15, f'Expected 15 rows, got {len(rows)}'
print('\n✅ Validation passed: exactly 15 rows')

## Cell 14 — Kaggle CLI Submit

Run this cell to submit directly from Kaggle!

In [ ]:
!kaggle competitions submit \
    -c niat-masterclass-rag-challenge \
    -f /kaggle/working/submission.csv \
    -m "Zyro Dynamics HR RAG - LLaMA3 + FAISS + MMR + Guardrails"